# 発展課題4（演習4：壊さないように測る）

演習4-3 で組んで測ったあと、**手を入れて測り直す**ところです。

- [問4-1 ボトルネックを潰したら、次はどこか](#scrollTo=adv04_measure_02)（実験）
- [問4-2 表示の値段を計算する](#scrollTo=adv04_measure_06)（紙と鉛筆）

> **最初に下のセル（`cq.h`）を1回だけ実行してください。**

In [ ]:
%%writefile cq.h
// ============================================================
//  cq.h ―― この演習で使うスレッドセーフなキュー
//  中身は読まなくてよい。使うのは push / pop / size の3つだけ。
// ============================================================
#pragma once
#include <queue>
#include <mutex>
#include <condition_variable>

template <typename T>
class ConcurrentQueue {
public:
    explicit ConcurrentQueue(std::size_t capacity) : capacity_(capacity) {}

    // 入れる。満杯なら空くまで待つ
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v);
        if (q_.size() > peak_) peak_ = q_.size();
        lk.unlock();
        can_pop_.notify_one();
    }

    // 取り出す。空なら来るまで待つ（失敗しない）
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop();
        lk.unlock();
        can_push_.notify_one();
        return v;
    }

    std::size_t size() const { std::lock_guard<std::mutex> g(mtx_); return q_.size(); }
    std::size_t peak() const { std::lock_guard<std::mutex> g(mtx_); return peak_; }   // 観測用：並んだ最大数

private:
    std::queue<T> q_;
    std::size_t capacity_;
    std::size_t peak_ = 0;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_;    // 「取り出せるようになった」
    std::condition_variable can_push_;   // 「入れられるようになった」
};

## 問4-1. ボトルネックを潰したら、次はどこか

演習4-3 では **30 FPS**、犯人は **Show**（正味 33ms、どこも待っていない）でした。
その **Show を 30ms → 10ms** に速くします。

```
Read  = read 13ms（CPU）
Infer = pre 9ms（CPU） + dpu 2ms（外部） + post 2ms（CPU） = 13ms
Show  = show 30ms  →  10ms
```

- (a) **FPS はいくつになるか。** 3倍の 90 FPS になるでしょうか
- (b) **犯人はどの段に移るか。** `取り出し待ち` / `正味` / `入れ待ち` の3列で見分けてください

> 30ms の結果も一緒に出ます。**2つの表を並べて比べて**ください。

In [ ]:
%%writefile adv04a.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <vector>
#include <chrono>
#include <atomic>
#include "cq.h"
using namespace std::chrono;

// ---- 仕事の中身（演習4-2 と同じ） ----
long calib = 0;
std::atomic<long> sink{0};        // 複数スレッドから足すので atomic（volatile では守れない）
long burn(long n) { long s = 0; for (long i = 0; i < n; i++) s += (i * 2654435761u) % 7; return s; }
void cpu_ms(int ms)  { sink.fetch_add(burn(calib * ms), std::memory_order_relaxed); }   // CPU が計算する区間
void ext_ms(int ms)  { std::this_thread::sleep_for(milliseconds(ms)); }   // 外部（DPU・画面）にまかせている区間
void calibrate() {
    long n = 100000;
    for (;;) {
        auto a = steady_clock::now(); sink.fetch_add(burn(n), std::memory_order_relaxed);
        auto us = duration_cast<microseconds>(steady_clock::now() - a).count();
        if (us > 30000) { calib = n * 1000 / us; break; }
        n *= 2;
    }
}

const int N = 33, WARMUP = 3;
const std::size_t CAP = 4;

// 段ごとの記録（各スレッドが自分専用に持つので、鍵は要らない）―― 演習4-3 と同じ
struct Stat { long long pop_us = 0, work_us = 0, push_us = 0; };

void run(int show_ms) {
    ConcurrentQueue<int> q1(CAP), q2(CAP);
    Stat sr, si, ss;
    auto t0 = steady_clock::now();
    steady_clock::time_point tstart;

    auto lap = [](steady_clock::time_point& t) {
        auto n = steady_clock::now();
        long long us = duration_cast<microseconds>(n - t).count();
        t = n; return us;
    };

    std::thread rd([&] {
        for (int i = 0; i < N; i++) {
            auto t = steady_clock::now();
            cpu_ms(13);                     sr.work_us += (i < WARMUP ? 0 : lap(t));   // read
            q1.push(i);                     sr.push_us += (i < WARMUP ? 0 : lap(t));   // 入れ待ち
        }
    });
    std::thread in([&] {
        for (int i = 0; i < N; i++) {
            auto t = steady_clock::now();
            int v = q1.pop();               si.pop_us  += (i < WARMUP ? 0 : lap(t));   // 取り出し待ち
            cpu_ms(9); ext_ms(2); cpu_ms(2);
                                            si.work_us += (i < WARMUP ? 0 : lap(t));   // pre+dpu+post
            q2.push(v);                     si.push_us += (i < WARMUP ? 0 : lap(t));   // 入れ待ち
        }
    });
    std::thread sh([&] {
        for (int i = 0; i < N; i++) {
            auto t = steady_clock::now();
            q2.pop();                       ss.pop_us  += (i < WARMUP ? 0 : lap(t));   // 取り出し待ち
            if (i == WARMUP) tstart = steady_clock::now();
            ext_ms(i % 10 == 0 ? show_ms * 2 : show_ms);
                                            ss.work_us += (i < WARMUP ? 0 : lap(t));   // show
        }
    });
    rd.join(); in.join(); sh.join();

    double sec = duration_cast<microseconds>(steady_clock::now() - tstart).count() / 1e6;
    int n = N - WARMUP;
    std::cout << "\n【Show = " << show_ms << "ms のとき】  実測 "
              << std::fixed << std::setprecision(1) << ((N - WARMUP) / sec) << " FPS\n";
    std::cout << "  取り出し待ち     正味     入れ待ち   段\n";
    auto line = [&](const char* nm, const Stat& s) {
        std::cout << std::setw(11) << std::setprecision(1) << (s.pop_us / 1000.0 / n) << "ms"
                  << std::setw(9)  << (s.work_us / 1000.0 / n) << "ms"
                  << std::setw(11) << (s.push_us / 1000.0 / n) << "ms   " << nm << "\n";
    };
    line("Read",  sr);
    line("Infer", si);
    line("Show",  ss);
}

int main() {
    calibrate();
    std::cout << "3段パイプライン（Read / Infer / Show を1人ずつ、容量" << CAP << "）\n"
              << "「待っていない段」がボトルネックです。\n";
    run(30);        // 演習4-3 と同じ条件
    run(10);        // Show を 3倍速くしてみる
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread -O2 adv04a.cpp -o adv04a && ./adv04a

### 問4-1 の解答 ―― 犯人は移る。そして次は2人いる

```
【Show = 30ms のとき】  実測 30.2 FPS
  取り出し待ち     正味     入れ待ち   段
        0.0ms     12.2ms       13.4ms   Read
        0.3ms     13.0ms       17.1ms   Infer
        0.0ms     33.1ms        0.0ms   Show     ← 犯人

【Show = 10ms のとき】  実測 72.8 FPS
  取り出し待ち     正味     入れ待ち   段
        0.0ms     13.5ms        0.0ms   Read     ← 犯人
        0.4ms     13.4ms        0.0ms   Infer    ← こちらも犯人
        2.5ms     11.3ms        0.0ms   Show     ← 待つ側にまわった
```

（`正味` は Colab の混み具合で上下します。**見るべきは3つの列の関係**です）

**(a) 約 73 FPS。** Read と Infer の 13ms より下には行けないので、
`1000 / 13.5 ≒ 74 FPS` が新しい上限A です。

> **一番遅い段を速くしても、次に遅い段までしか速くならない。**

**(b) 犯人は Read と Infer に移りました。**

- **Show**: `取り出し待ち` 0.0 → **2.5ms**。データを待つ側になりました
- **Read / Infer**: `入れ待ち` 13〜17ms → **0.0ms**。下流がすぐ受け取るので詰まりません
- 残ったのは `正味` だけ ―― **どこでも待っていない**状態です

**犯人は2人います。** Read も Infer も 13ms なので、**片方だけ速くしても FPS は上がりません。**

> **ボトルネックは移動する。潰したら必ず測り直す。次が1つとは限らない。**

## 問4-2. 表示の値段を計算する

KV260 をシリアルコンソール（**115200 baud**）でつないで動かしています。
`cout` で1行 36 バイトのデバッグ表示を出すとします。

- (a) 1行を出すのに何 ms かかるか
- (b) 1フレームにつき**4行**出すと、1フレームあたり何 ms 増えるか
- (c) **1フレーム 82ms の処理**と **1フレーム 13ms の処理**で、それぞれ何%の増加になるか
- (d) この表示を**すべて計測区間の外**に置いた場合、
  各段の測定値（pre / dpu / post）は正しい値を出し続けるか。
  **では、どうやってこの増加に気づけばよいか**

### 問4-2 の解答

**(a)** 115200 baud は 1バイト 10ビット（スタート/ストップビット込み）で送るので
**毎秒 11520 バイト**、1バイト **約 87µs**。36バイトの行で `36 × 87µs ≒` **3.1ms**。

**(b)** `4行 × 3.1ms =` **12.4ms/フレーム**。

**(c)**

- 1フレーム 82ms の処理 … 12.4 / 82 ≒ **15%**
- 1フレーム 13ms の処理 … 12.4 / 13 ≒ **95%**。**処理時間とほぼ同じだけ増えます**

> **処理を軽くするほど、表示のコストが相対的に効いてきます。**

**(d) 正しい値を出し続けます。** 表示は計測区間の外なので、`pre` も `dpu` も `post` も
もっともらしい数字のままです。それなのに全体は 12ms/フレーム遅い ―― **これがいちばん怖い状態です。**

> **「各段の合計」と「1フレームの実測時間」を並べて出し、差（＝計測外）を見る。**

演習4-2 の表の最後の行がそれです。ここが `0.0ms` でないなら、
**他のどの数字も信用してはいけません。**

対策は演習4-2 の作法どおりです。**実行中は足し込むだけ。表示は全部終わってから1回だけ。**

ただし「1回だけ」は、**貯めたログを最後に全部流す**という意味ではありません。
400行をシリアルに流せば、それだけで 1.2秒かかります。出すのは段ごとの平均・最大・計測外だけ ――
演習4-2 の `Profiler::report()` が出している6〜7行です。